# Self-Reflective and Continually-Learning AI Assistant for Crop Disease Diagnosis

**PBL-III research notebook — final experimental pipeline**

This notebook trains and evaluates three crop-disease classifiers, then investigates whether confidence, predictive entropy, and feature-space familiarity can identify potentially unreliable predictions.

**Current implementation:** an experimental reliability-assessment prototype. It flags potentially unreliable predictions for review; it does not correct predicted classes, learn from feedback, or implement continual learning.

## Final classification results

| Architecture | Mean OOF accuracy | Fold standard deviation | OOF errors |
|---|---:|---:|---:|
| MobileNetV3-Small | 93.3279% | 0.1395 percentage points | 1,377 |
| ResNet18 | 93.3327% | 0.3949 percentage points | 1,376 |
| **EfficientNet-B0** | **94.6555%** | 0.4022 percentage points | **1,103** |

The reported values are from the revised three-fold experiments. They supersede the earlier model results in the repository.

## Main reliability findings

- Low confidence was the strongest standalone signal for ranking incorrect predictions.
- Predictive entropy was highly correlated with confidence (Spearman correlation approximately 0.9986).
- Feature-space familiarity was comparatively weak as a standalone error-ranking signal.
- Combining the signals did not consistently improve reliability assessment over the confidence baseline.
- The paired bootstrap analysis did not establish a statistically significant AURC improvement.

## Execution environment and inputs

The notebook is organized for **Kaggle Notebooks** and expects the PlantVillage dataset to be attached as a Kaggle input. A GPU is strongly recommended because the training section retrains three models across three folds.

Run cells from top to bottom in a fresh kernel. The model-training section is computationally expensive. The analysis sections use the OOF predictions and checkpoints generated by the training section.

The dataset and trained checkpoints are not stored in this GitHub repository. Only selected, lightweight evaluation artifacts are published under `results/`.

## 1. Environment, paths, and imports

The dataset location is configured once below. Change `DATASET_DIR` if the attached Kaggle dataset is mounted elsewhere. Experiment outputs are written to `/kaggle/working/plant_disease_experiments/`.

In [ ]:
from pathlib import Path
import gc
import random
import json
import math
import sys
import subprocess
import zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

from PIL import Image
from tqdm.auto import tqdm
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

# Kaggle input and output paths.
DATASET_DIR = Path(
    "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage"
)
PROJECT_DIR = Path("/kaggle/working/plant_disease_experiments")
RESULTS_DIR = PROJECT_DIR / "results"
MODELS_DIR = PROJECT_DIR / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else device)
print("Dataset path:", DATASET_DIR)
print("Project output:", PROJECT_DIR)

## 2. Dataset loading and verification

The experiment uses a 15-class subset of PlantVillage covering Pepper, Potato, and Tomato. Images are resized to 224 × 224 and normalized with mean and standard deviation of 0.5 per channel.

In [ ]:
# The experiment uses the same 15-class Pepper, Potato, and Tomato subset.
EXPECTED_CLASSES = {
    "Pepper__bell___Bacterial_spot",
    "Pepper__bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot",
    "Tomato_Spider_mites_Two_spotted_spider_mite",
    "Tomato__Target_Spot",
    "Tomato__Tomato_YellowLeaf__Curl_Virus",
    "Tomato__Tomato_mosaic_virus",
    "Tomato_healthy",
}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3),
])

assert DATASET_DIR.exists(), (
    f"PlantVillage dataset not found at {DATASET_DIR}. "
    "Attach the dataset to the Kaggle notebook or update DATASET_DIR."
)

actual_classes = {
    path.name for path in DATASET_DIR.iterdir() if path.is_dir()
}
assert actual_classes == EXPECTED_CLASSES, (
    f"Dataset class folders differ. Missing: {EXPECTED_CLASSES - actual_classes}; "
    f"extra: {actual_classes - EXPECTED_CLASSES}"
)

dataset = datasets.ImageFolder(DATASET_DIR, transform=transform)
num_classes = len(dataset.classes)

assert len(dataset) == 20_638
assert num_classes == 15
assert set(dataset.classes) == EXPECTED_CLASSES

class_counts = pd.Series(dataset.targets).value_counts().sort_index()
class_summary = pd.DataFrame({
    "Class": dataset.classes,
    "Images": [int(class_counts.get(i, 0)) for i in range(num_classes)],
})

print(f"Dataset verification passed: {len(dataset):,} images, {num_classes} classes.")
display(class_summary)

## 3. Three-fold stratified cross-validation

All three architectures use the same dataset ordering and three-fold stratified split (`random_state=42`). Each image receives an out-of-fold prediction from a checkpoint that did not train on that image.

In [ ]:
kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
fold_results = []

## 4. MobileNetV3-Small training

The ImageNet-pretrained MobileNetV3-Small backbone parameters are frozen and a new 15-class linear output layer is trained for 10 epochs in each fold. The fold checkpoints and OOF logits/predictions are saved under the experiment output directory.

In [ ]:
from tqdm.auto import tqdm

# Fresh OOF storage because the previous run was interrupted
oof_logits = np.zeros((len(dataset), num_classes), dtype=np.float32)
oof_labels = np.array(dataset.targets)
oof_predictions = np.zeros(len(dataset), dtype=int)
oof_folds = np.zeros(len(dataset), dtype=int)

fold_results = []

best_accuracy = 0
best_model = None
best_fold = None

# Faster data loading without changing the experiment
NUM_WORKERS = 4

for fold, (train_idx, val_idx) in enumerate(
    kf.split(np.zeros(len(dataset)), dataset.targets)
):

    print(f"\n===== Fold {fold + 1} =====", flush=True)

    train_dataset = Subset(dataset, train_idx)
    val_dataset = Subset(dataset, val_idx)

    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=True
    )

    model = models.mobilenet_v3_small(weights="DEFAULT")

    for param in model.parameters():
        param.requires_grad = False

    model.classifier[3] = nn.Linear(
        model.classifier[3].in_features,
        num_classes
    )

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.classifier[3].parameters(),
        lr=0.001
    )

    epochs = 10

    # Training
    for epoch in range(epochs):

        model.train()
        running_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"Fold {fold+1} | Epoch {epoch+1}/{epochs}"
        )

        for images, labels in progress:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        print(
            f"Epoch {epoch+1} completed | "
            f"Total Loss: {running_loss:.4f}",
            flush=True
        )

    # Validation and OOF collection
    model.eval()

    all_preds = []
    all_labels = []

    correct = 0
    total = 0
    val_position = 0

    with torch.no_grad():

        for images, labels in tqdm(
            val_loader,
            desc=f"Fold {fold+1} | Validation"
        ):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            predicted = torch.argmax(outputs, dim=1)

            predictions_cpu = predicted.cpu().numpy()
            labels_cpu = labels.cpu().numpy()
            logits_cpu = outputs.cpu().numpy()

            all_preds.extend(predictions_cpu)
            all_labels.extend(labels_cpu)

            batch_size = labels.size(0)

            batch_indices = val_idx[
                val_position:val_position + batch_size
            ]

            oof_logits[batch_indices] = logits_cpu
            oof_predictions[batch_indices] = predictions_cpu
            oof_folds[batch_indices] = fold + 1

            val_position += batch_size

            total += batch_size
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    fold_results.append(accuracy)

    print(
        f"Fold {fold+1} Accuracy: {accuracy:.2f}%",
        flush=True
    )

    # Save fold checkpoint
    torch.save(
        model.state_dict(),
        MODELS_DIR / f"mobilenet_fold_{fold+1}.pth"
    )

    if fold == 0 or accuracy > best_accuracy:

        best_accuracy = accuracy
        best_model = model
        best_fold = fold + 1

        torch.save(
            model.state_dict(),
            MODELS_DIR / "best_model.pth"
        )

    # Save OOF progress after each completed fold
    np.savez_compressed(
        RESULTS_DIR / "baseline_oof_partial.npz",
        oof_logits=oof_logits,
        oof_labels=oof_labels,
        oof_predictions=oof_predictions,
        oof_folds=oof_folds,
        fold_results=np.array(fold_results),
        class_names=np.array(dataset.classes)
    )

    print(
        f"Fold {fold+1} checkpoint and OOF progress saved.",
        flush=True
    )

    del train_loader, val_loader
    del train_dataset, val_dataset

# Final results
print("\n===== BASELINE COMPLETE =====")

print("Fold accuracies:", fold_results)
print("Mean accuracy:", np.mean(fold_results))
print("Standard deviation:", np.std(fold_results))
print("Best fold:", best_fold)
print("Best accuracy:", best_accuracy)

### 4.1 Validate and save MobileNetV3-Small OOF predictions

This cell checks OOF completeness, verifies that stored predictions match the logits, checks the fold accuracies, and saves the baseline OOF artifact.

In [ ]:
import numpy as np
import pandas as pd

# 1. Verify OOF completeness
assert oof_logits.shape == (20638, 15)
assert oof_labels.shape == (20638,)
assert oof_predictions.shape == (20638,)
assert oof_folds.shape == (20638,)

assert np.isfinite(oof_logits).all()
assert np.all(oof_folds >= 1)
assert np.all(oof_folds <= 3)
assert set(np.unique(oof_folds)) == {1, 2, 3}

assert np.array_equal(oof_labels, np.array(dataset.targets))
assert np.array_equal(
    oof_predictions,
    np.argmax(oof_logits, axis=1)
)

# 2. Verify fold-level accuracies from OOF predictions
fold_counts = np.bincount(oof_folds)[1:]

verified_accuracies = []

for fold in range(1, 4):
    mask = oof_folds == fold
    acc = np.mean(
        oof_predictions[mask] == oof_labels[mask]
    ) * 100
    verified_accuracies.append(acc)

print("Fold sample counts:", fold_counts)
print("Recorded fold accuracies:", fold_results)
print("OOF verified accuracies:", verified_accuracies)
print(
    "OOF overall accuracy:",
    round(np.mean(oof_predictions == oof_labels) * 100, 4)
)

assert np.allclose(
    verified_accuracies,
    fold_results,
    atol=0.01
)

# 3. Save final OOF artifacts
np.savez_compressed(
    RESULTS_DIR / "baseline_oof.npz",
    oof_logits=oof_logits,
    oof_labels=oof_labels,
    oof_predictions=oof_predictions,
    oof_folds=oof_folds,
    class_names=np.array(dataset.classes),
    fold_accuracies=np.array(fold_results)
)

oof_df = pd.DataFrame({
    "image_index": np.arange(len(dataset)),
    "fold": oof_folds,
    "true_label": oof_labels,
    "predicted_label": oof_predictions,
    "true_class": [dataset.classes[i] for i in oof_labels],
    "predicted_class": [
        dataset.classes[i] for i in oof_predictions
    ],
    "correct": oof_labels == oof_predictions
})

oof_df.to_csv(
    RESULTS_DIR / "baseline_oof.csv",
    index=False
)

# 4. Verify checkpoint files
for fold in range(1, 4):
    checkpoint = MODELS_DIR / f"mobilenet_fold_{fold}.pth"
    assert checkpoint.exists(), f"Missing checkpoint: {checkpoint}"
    print("Checkpoint:", checkpoint.name,
          f"({checkpoint.stat().st_size / 1024**2:.2f} MB)")

print("\nOOF VERIFICATION AND SAVING COMPLETE")

for path in [
    RESULTS_DIR / "baseline_oof.npz",
    RESULTS_DIR / "baseline_oof.csv"
]:
    print(path.name, f"({path.stat().st_size / 1024**2:.2f} MB)")

## 5. ResNet18 and EfficientNet-B0 training

The same three-fold protocol is used for the two comparison architectures. Their ImageNet-pretrained parameters are frozen and only the new final classifier layers are optimized.

In [ ]:
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Subset
from torchvision import models
from tqdm.auto import tqdm

def build_comparison_model(architecture, num_classes):

    if architecture == "resnet18":

        model = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )

        for param in model.parameters():
            param.requires_grad = False

        model.fc = nn.Linear(
            model.fc.in_features,
            num_classes
        )

        trainable_params = model.fc.parameters()

    elif architecture == "efficientnet_b0":

        model = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.DEFAULT
        )

        for param in model.parameters():
            param.requires_grad = False

        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features,
            num_classes
        )

        trainable_params = model.classifier[1].parameters()

    else:
        raise ValueError(f"Unknown architecture: {architecture}")

    return model, trainable_params


def train_comparison_model(architecture):

    print(f"\n{'='*60}")
    print(f"TRAINING: {architecture.upper()}")
    print(f"{'='*60}")

    architecture_results = RESULTS_DIR / architecture
    architecture_models = MODELS_DIR / architecture

    architecture_results.mkdir(parents=True, exist_ok=True)
    architecture_models.mkdir(parents=True, exist_ok=True)

    n = len(dataset)

    model_oof_logits = np.zeros(
        (n, num_classes), dtype=np.float32
    )
    model_oof_predictions = np.zeros(n, dtype=np.int64)
    model_oof_folds = np.zeros(n, dtype=np.int64)
    model_oof_labels = np.asarray(dataset.targets)

    fold_accuracies = []

    for fold_idx, (train_idx, val_idx) in enumerate(
        kf.split(np.zeros(n), dataset.targets)
    ):

        fold = fold_idx + 1

        print(f"\n===== {architecture} | Fold {fold} =====", flush=True)

        # Reproducible initialization and shuffled data order
        seed = 2026 + fold
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        generator = torch.Generator()
        generator.manual_seed(seed)

        train_dataset = Subset(dataset, train_idx)
        val_dataset = Subset(dataset, val_idx)

        train_loader = DataLoader(
            train_dataset,
            batch_size=32,
            shuffle=True,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True,
            generator=generator
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=32,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
            persistent_workers=True
        )

        model, trainable_params = build_comparison_model(
            architecture, num_classes
        )

        model = model.to(device)

        criterion = nn.CrossEntropyLoss()

        optimizer = optim.Adam(
            trainable_params,
            lr=0.001
        )

        epochs = 10

        # Training
        for epoch in range(epochs):

            model.train()
            running_loss = 0.0

            progress = tqdm(
                train_loader,
                desc=f"{architecture} | Fold {fold} | Epoch {epoch+1}/{epochs}"
            )

            for images, labels in progress:

                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                outputs = model(images)
                loss = criterion(outputs, labels)

                loss.backward()
                optimizer.step()

                running_loss += loss.item()

                progress.set_postfix(
                    loss=f"{loss.item():.4f}"
                )

            print(
                f"Epoch {epoch+1}/10 | "
                f"Average loss: {running_loss / len(train_loader):.4f}",
                flush=True
            )

        # Validation and OOF collection
        model.eval()

        correct = 0
        total = 0
        val_position = 0

        with torch.no_grad():

            for images, labels in tqdm(
                val_loader,
                desc=f"{architecture} | Fold {fold} | Validation"
            ):

                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                outputs = model(images)
                predictions = outputs.argmax(dim=1)

                logits_cpu = outputs.cpu().numpy()
                predictions_cpu = predictions.cpu().numpy()

                batch_size = len(labels)

                batch_indices = val_idx[
                    val_position:val_position + batch_size
                ]

                model_oof_logits[batch_indices] = logits_cpu
                model_oof_predictions[batch_indices] = predictions_cpu
                model_oof_folds[batch_indices] = fold

                val_position += batch_size

                correct += (predictions == labels).sum().item()
                total += batch_size

        accuracy = 100 * correct / total
        fold_accuracies.append(accuracy)

        print(
            f"{architecture} Fold {fold} accuracy: {accuracy:.4f}%",
            flush=True
        )

        # Save fold checkpoint
        checkpoint_path = (
            architecture_models / f"{architecture}_fold_{fold}.pth"
        )

        torch.save(model.state_dict(), checkpoint_path)

        # Save partial OOF artifacts after every fold
        np.savez_compressed(
            architecture_results / "oof_partial.npz",
            logits=model_oof_logits,
            labels=model_oof_labels,
            predictions=model_oof_predictions,
            folds=model_oof_folds,
            fold_accuracies=np.array(fold_accuracies),
            class_names=np.array(dataset.classes)
        )

        print("Fold checkpoint and partial OOF data saved.")

        del model, optimizer, criterion
        del train_loader, val_loader
        del train_dataset, val_dataset

        gc.collect()
        torch.cuda.empty_cache()

    # Final verification
    assert np.all(np.isfinite(model_oof_logits))
    assert set(np.unique(model_oof_folds)) == {1, 2, 3}
    assert np.array_equal(
        model_oof_labels,
        np.asarray(dataset.targets)
    )

    assert np.array_equal(
        model_oof_predictions,
        np.argmax(model_oof_logits, axis=1)
    )

    overall_accuracy = np.mean(
        model_oof_predictions == model_oof_labels
    ) * 100

    # Save final OOF artifacts
    np.savez_compressed(
        architecture_results / f"{architecture}_oof.npz",
        logits=model_oof_logits,
        labels=model_oof_labels,
        predictions=model_oof_predictions,
        folds=model_oof_folds,
        fold_accuracies=np.array(fold_accuracies),
        class_names=np.array(dataset.classes)
    )

    pd.DataFrame({
        "image_index": np.arange(n),
        "fold": model_oof_folds,
        "true_label": model_oof_labels,
        "predicted_label": model_oof_predictions,
        "correct": model_oof_labels == model_oof_predictions
    }).to_csv(
        architecture_results / f"{architecture}_oof.csv",
        index=False
    )

    print(f"\n===== {architecture.upper()} COMPLETE =====")
    print("Fold accuracies:", fold_accuracies)
    print("Mean accuracy:", np.mean(fold_accuracies))
    print("Fold standard deviation:", np.std(fold_accuracies))
    print("Overall OOF accuracy:", overall_accuracy)

    return {
        "architecture": architecture,
        "fold_accuracies": fold_accuracies,
        "mean_accuracy": np.mean(fold_accuracies),
        "std_accuracy": np.std(fold_accuracies),
        "oof_accuracy": overall_accuracy
    }

In [ ]:
resnet18_results = train_comparison_model("resnet18")

In [ ]:
efficientnet_results = train_comparison_model("efficientnet_b0")

## 6. Class-wise performance and confusion analysis

The following cells summarize OOF class-wise precision, recall, F1, support, and confusion matrices for all three architectures. The confusion analysis reports the most frequent directional misclassifications.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    precision_recall_fscore_support,
    confusion_matrix
)

RESULTS_DIR = PROJECT_DIR / "results"

MODEL_FILES = {
    "MobileNetV3-Small": RESULTS_DIR / "baseline_oof.npz",
    "ResNet18": RESULTS_DIR / "resnet18" / "resnet18_oof.npz",
    "EfficientNet-B0": RESULTS_DIR / "efficientnet_b0" / "efficientnet_b0_oof.npz",
}

def get_array(archive, candidates):
    for key in candidates:
        if key in archive.files:
            return archive[key]
    raise KeyError(
        f"None of {candidates} found. Available keys: {archive.files}"
    )

classwise_rows = []
foldwise_rows = []
confusion_rows = []

reference_classes = None

for model_name, file_path in MODEL_FILES.items():
    print(f"\nProcessing {model_name}...")

    with np.load(file_path, allow_pickle=False) as z:
        y_true = get_array(z, ["oof_labels", "labels"]).astype(int)
        y_pred = get_array(z, ["oof_predictions", "predictions"]).astype(int)
        folds = get_array(z, ["oof_folds", "folds"]).astype(int)
        classes = get_array(z, ["class_names"])

    classes = [str(x) for x in classes]

    assert len(y_true) == len(y_pred) == len(folds) == 20638
    assert len(classes) == 15

    if reference_classes is None:
        reference_classes = classes
    else:
        assert classes == reference_classes, (
            f"Class ordering mismatch in {model_name}"
        )

    labels = np.arange(len(classes))

    # Overall out-of-fold class-wise metrics
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        zero_division=0
    )

    for i, class_name in enumerate(classes):
        errors = int(np.sum((y_true == i) & (y_pred != i)))

        classwise_rows.append({
            "Architecture": model_name,
            "Class": class_name,
            "Support": int(support[i]),
            "Errors": errors,
            "Error_Rate": errors / support[i] if support[i] else np.nan,
            "Precision": precision[i],
            "Recall": recall[i],
            "F1": f1[i],
        })

    # Fold-wise class performance
    for fold in sorted(np.unique(folds)):
        mask = folds == fold

        p, r, f, s = precision_recall_fscore_support(
            y_true[mask],
            y_pred[mask],
            labels=labels,
            zero_division=0
        )

        for i, class_name in enumerate(classes):
            foldwise_rows.append({
                "Architecture": model_name,
                "Fold": int(fold),
                "Class": class_name,
                "Support": int(s[i]),
                "Precision": p[i],
                "Recall": r[i],
                "F1": f[i],
            })

    # Confusion matrix in long format
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    for true_idx in range(len(classes)):
        for pred_idx in range(len(classes)):
            confusion_rows.append({
                "Architecture": model_name,
                "True_Class": classes[true_idx],
                "Predicted_Class": classes[pred_idx],
                "Count": int(cm[true_idx, pred_idx]),
            })

    macro_p, macro_r, macro_f, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    print(f"OOF accuracy: {np.mean(y_true == y_pred):.4%}")
    print(f"Macro precision: {macro_p:.4f}")
    print(f"Macro recall:    {macro_r:.4f}")
    print(f"Macro F1:        {macro_f:.4f}")

# Save outputs
classwise_df = pd.DataFrame(classwise_rows)
foldwise_df = pd.DataFrame(foldwise_rows)
confusion_df = pd.DataFrame(confusion_rows)

classwise_df.to_csv(
    RESULTS_DIR / "classwise_oof_performance.csv", index=False
)
foldwise_df.to_csv(
    RESULTS_DIR / "classwise_oof_by_fold.csv", index=False
)
confusion_df.to_csv(
    RESULTS_DIR / "oof_confusion_matrices.csv", index=False
)

print("\nCLASS-WISE ERRORS — HIGHEST ERROR RATE FIRST")
display(
    classwise_df.sort_values(
        ["Architecture", "Error_Rate"],
        ascending=[True, False]
    ).round(4)
)

print("\nSAVED:")
print(RESULTS_DIR / "classwise_oof_performance.csv")
print(RESULTS_DIR / "classwise_oof_by_fold.csv")
print(RESULTS_DIR / "oof_confusion_matrices.csv")

In [ ]:
import pandas as pd
from pathlib import Path

RESULTS_DIR = PROJECT_DIR / "results"

cm_df = pd.read_csv(RESULTS_DIR / "oof_confusion_matrices.csv")

# Remove correct predictions
errors_df = cm_df[
    (cm_df["True_Class"] != cm_df["Predicted_Class"]) &
    (cm_df["Count"] > 0)
].copy()

# Calculate percentage of each true class assigned to each wrong class
class_support = (
    cm_df.groupby(["Architecture", "True_Class"])["Count"]
    .sum()
    .rename("Class_Support")
    .reset_index()
)

errors_df = errors_df.merge(
    class_support,
    on=["Architecture", "True_Class"]
)

errors_df["Error_Percentage"] = (
    100 * errors_df["Count"] / errors_df["Class_Support"]
)

# Top 10 confusions for each architecture
top_confusions = (
    errors_df.sort_values(
        ["Architecture", "Count"],
        ascending=[True, False]
    )
    .groupby("Architecture")
    .head(10)
    .reset_index(drop=True)
)

print("TOP 10 WRONG CLASSIFICATIONS PER ARCHITECTURE")
display(top_confusions.round(2))

# Identify directional confusions shared by all three models
shared = (
    errors_df.groupby(["True_Class", "Predicted_Class"])
    .agg(
        Architectures=("Architecture", "nunique"),
        Total_Errors=("Count", "sum")
    )
    .reset_index()
)

shared = shared[shared["Architectures"] == 3]
shared = shared.sort_values("Total_Errors", ascending=False)

print("\nCONFUSIONS COMMON TO ALL THREE ARCHITECTURES")
display(shared.head(20))

top_confusions.to_csv(
    RESULTS_DIR / "top_class_confusions.csv", index=False
)

shared.to_csv(
    RESULTS_DIR / "shared_class_confusions.csv", index=False
)

print("\nSaved confusion analysis.")

## 7. Corrected feature-space familiarity

For each architecture and outer fold:

1. The corresponding fold-specific checkpoint extracts embeddings for the dataset.
2. Class prototypes are calculated only from the training portion of that fold.
3. Held-out image embeddings are compared with the prototype for the model-predicted class.
4. The same checkpoint is used to regenerate held-out predictions; the notebook requires zero mismatches against the stored OOF predictions.

The familiarity distance is defined as \(S(x)=1-\cos(f(x),\mu_{\hat y})\). Larger values indicate lower familiarity. This is a feature-space similarity signal, not a calibrated probability of correctness or a validated OOD detector.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.preprocessing import normalize
from sklearn.metrics import roc_auc_score, average_precision_score

# ============================================================
# 1. PATHS
# ============================================================

# Use the OOF predictions and fold checkpoints generated in this notebook.
OUTPUT_DIR = RESULTS_DIR / "familiarity_recomputed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. DATASET
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * 3, [0.5] * 3)
])

dataset = datasets.ImageFolder(
    root=str(DATASET_DIR),
    transform=transform
)

assert len(dataset) == 20638
assert len(dataset.classes) == 15

print("Dataset:", len(dataset))
print("Classes:", len(dataset.classes))

# ============================================================
# 3. ARCHITECTURE CONFIGURATION
# ============================================================

CONFIGS = {
    "MobileNetV3-Small": {
        "oof": RESULTS_DIR / "baseline_oof.npz",
        "checkpoint_dir": MODELS_DIR,
        "checkpoint_pattern": "mobilenet_fold_{}.pth",
    },
    "ResNet18": {
        "oof": RESULTS_DIR / "resnet18" / "resnet18_oof.npz",
        "checkpoint_dir": MODELS_DIR / "resnet18",
        "checkpoint_pattern": "resnet18_fold_{}.pth",
    },
    "EfficientNet-B0": {
        "oof": RESULTS_DIR / "efficientnet_b0" / "efficientnet_b0_oof.npz",
        "checkpoint_dir": MODELS_DIR / "efficientnet_b0",
        "checkpoint_pattern": "efficientnet_b0_fold_{}.pth",
    },
}

# ============================================================
# 4. OOF LOADING
# ============================================================

def load_oof(path, architecture):
    with np.load(path, allow_pickle=False) as z:

        if architecture == "MobileNetV3-Small":
            labels = z["oof_labels"].astype(np.int64)
            predictions = z["oof_predictions"].astype(np.int64)
            folds = z["oof_folds"].astype(np.int64)
            classes = z["class_names"]

        else:
            labels = z["labels"].astype(np.int64)
            predictions = z["predictions"].astype(np.int64)
            folds = z["folds"].astype(np.int64)
            classes = z["class_names"]

    return labels, predictions, folds, [str(c) for c in classes]


# ============================================================
# 5. MODEL CONSTRUCTION
# ============================================================

def build_model(architecture):

    if architecture == "MobileNetV3-Small":
        model = models.mobilenet_v3_small(weights=None)
        model.classifier[3] = nn.Linear(
            model.classifier[3].in_features, 15
        )

    elif architecture == "ResNet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 15)

    elif architecture == "EfficientNet-B0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(
            model.classifier[1].in_features, 15
        )

    else:
        raise ValueError(architecture)

    return model


def load_checkpoint(model, checkpoint_path):

    try:
        state = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )
    except TypeError:
        state = torch.load(
            checkpoint_path,
            map_location=device
        )

    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]

    elif isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]

    model.load_state_dict(state, strict=True)

    return model.to(device).eval()


# ============================================================
# 6. EXTRACT EMBEDDING + LOGITS
# ============================================================

def forward_with_embedding(model, images, architecture):

    if architecture == "MobileNetV3-Small":
        x = model.features(images)
        x = model.avgpool(x)
        embedding = torch.flatten(x, 1)
        logits = model.classifier(embedding)

    elif architecture == "ResNet18":
        x = model.conv1(images)
        x = model.bn1(x)
        x = model.relu(x)
        x = model.maxpool(x)
        x = model.layer1(x)
        x = model.layer2(x)
        x = model.layer3(x)
        x = model.layer4(x)
        x = model.avgpool(x)

        embedding = torch.flatten(x, 1)
        logits = model.fc(embedding)

    elif architecture == "EfficientNet-B0":
        x = model.features(images)
        x = model.avgpool(x)
        embedding = torch.flatten(x, 1)
        logits = model.classifier(embedding)

    else:
        raise ValueError(architecture)

    return embedding, logits


def extract_all_embeddings(model, architecture):

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    embeddings = []
    logits_list = []

    with torch.inference_mode():

        for images, _ in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            emb, logits = forward_with_embedding(
                model,
                images,
                architecture
            )

            embeddings.append(emb.cpu().numpy())
            logits_list.append(logits.cpu().numpy())

    return (
        np.concatenate(embeddings, axis=0),
        np.concatenate(logits_list, axis=0)
    )


# ============================================================
# 7. CORRECTED FAMILIARITY EXTRACTION
# ============================================================

summary_rows = []

for architecture, config in CONFIGS.items():

    print("\n" + "=" * 65)
    print("ARCHITECTURE:", architecture)
    print("=" * 65)

    labels, predictions, folds, class_names = load_oof(
        config["oof"],
        architecture
    )

    assert len(labels) == len(dataset)
    assert len(predictions) == len(dataset)
    assert len(folds) == len(dataset)
    assert class_names == dataset.classes
    assert np.array_equal(
        labels,
        np.asarray(dataset.targets)
    )

    corrected_familiarity = np.full(
        len(dataset),
        np.nan,
        dtype=np.float64
    )

    for fold in [1, 2, 3]:

        print(f"\nFold {fold}: loading checkpoint...")

        checkpoint_path = (
            config["checkpoint_dir"] /
            config["checkpoint_pattern"].format(fold)
        )

        assert checkpoint_path.exists(), checkpoint_path

        model = build_model(architecture)
        model = load_checkpoint(model, checkpoint_path)

        # Same checkpoint extracts training and validation
        # embeddings, ensuring a common feature space.
        all_embeddings, all_logits = extract_all_embeddings(
            model,
            architecture
        )

        train_idx = np.where(folds != fold)[0]
        val_idx = np.where(folds == fold)[0]

        print("Training samples:", len(train_idx))
        print("Held-out samples:", len(val_idx))

        # ----------------------------------------------------
        # Verify this checkpoint reproduces saved OOF output
        # ----------------------------------------------------

        reproduced_predictions = np.argmax(
            all_logits[val_idx],
            axis=1
        )

        mismatch_count = int(np.sum(
            reproduced_predictions != predictions[val_idx]
        ))

        print("Prediction mismatches:", mismatch_count)

        assert mismatch_count == 0, (
            f"{architecture}, fold {fold}: OOF reproduction failed. "
            "Stopping before familiarity calculation."
        )

        # ----------------------------------------------------
        # Build prototypes from this SAME checkpoint's
        # training embeddings.
        # ----------------------------------------------------

        prototypes = np.zeros(
            (len(class_names), all_embeddings.shape[1]),
            dtype=np.float64
        )

        for class_id in range(len(class_names)):

            class_indices = train_idx[
                labels[train_idx] == class_id
            ]

            assert len(class_indices) > 0

            prototypes[class_id] = np.mean(
                all_embeddings[class_indices],
                axis=0
            )

        normalized_prototypes = normalize(prototypes)

        normalized_validation = normalize(
            all_embeddings[val_idx]
        )

        similarities = (
            normalized_validation @ normalized_prototypes.T
        )

        predicted_classes = predictions[val_idx]

        corrected_familiarity[val_idx] = similarities[
            np.arange(len(val_idx)),
            predicted_classes
        ]

        print(
            "Mean familiarity:",
            round(
                float(corrected_familiarity[val_idx].mean()),
                5
            )
        )

        del model, all_embeddings, all_logits

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Save and evaluate
    # --------------------------------------------------------

    assert np.isfinite(corrected_familiarity).all()

    low_familiarity = 1.0 - corrected_familiarity
    errors = (predictions != labels).astype(np.int64)

    auc = roc_auc_score(errors, low_familiarity)
    ap = average_precision_score(errors, low_familiarity)

    filename = (
        architecture.lower()
        .replace("-", "_")
        .replace(" ", "_")
        + "_corrected.npz"
    )

    output_file = OUTPUT_DIR / filename

    np.savez_compressed(
        output_file,
        familiarity=corrected_familiarity,
        low_familiarity=low_familiarity,
        labels=labels,
        predictions=predictions,
        folds=folds,
        errors=errors,
        class_names=np.asarray(class_names)
    )

    summary_rows.append({
        "Architecture": architecture,
        "Samples": len(labels),
        "Errors": int(errors.sum()),
        "Familiarity_mean": float(corrected_familiarity.mean()),
        "Familiarity_correct_mean": float(
            corrected_familiarity[errors == 0].mean()
        ),
        "Familiarity_error_mean": float(
            corrected_familiarity[errors == 1].mean()
        ),
        "Low_familiarity_AUROC": float(auc),
        "Low_familiarity_AP": float(ap),
        "Output": str(output_file)
    })

    print("\nSaved:", output_file)
    print(f"Low-familiarity AUROC: {auc:.4f}")
    print(f"Low-familiarity AP:    {ap:.4f}")


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

summary_df = pd.DataFrame(summary_rows)

summary_file = (
    OUTPUT_DIR / "corrected_familiarity_summary.csv"
)

summary_df.to_csv(summary_file, index=False)

print("\n" + "=" * 65)
print("CORRECTED FAMILIARITY EXTRACTION COMPLETE")
print("=" * 65)

display(summary_df.round(5))

print("\nSummary saved:", summary_file)

## 8. Corrected reflection evaluation

The evaluation uses three signals:

- \(C=1-\max_k p_k\): low confidence.
- \(H=-\sum_k p_k\log(p_k)/\log(15)\): normalized predictive entropy.
- \(S=1-\cos(f(x),\mu_{\hat y})\): low familiarity.

The direct confidence baseline is compared with logistic-regression risk models using individual signals and their combinations. Within each outer fold, each split seed creates meta-training, threshold-calibration, and held-out test partitions. Models are fit on meta-training data, thresholds are selected using calibration errors only, and final metrics are measured on the untouched test partition.

The next cell exports fold-level results, test predictions, calibration checks, split manifests, and pooled metrics.

In [ ]:
# ================================================================
# PBL-3: CORRECTED REFLECTION EVALUATION V3
# Uses corrected, fold-consistent familiarity scores.
# No CNN retraining.
# ================================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------
# 1. Paths and experiment configuration
# ------------------------------------------------

WORK_ROOT = PROJECT_DIR
RESULTS_DIR = WORK_ROOT / "results"
CORRECTED_DIR = RESULTS_DIR / "familiarity_recomputed"

OUTPUT_DIR = RESULTS_DIR / "evaluation_v3_corrected"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARCHITECTURES = {
    "MobileNetV3-Small": {
        "oof": RESULTS_DIR / "baseline_oof.npz",
        "familiarity": CORRECTED_DIR / "mobilenetv3_small_corrected.npz",
    },
    "ResNet18": {
        "oof": RESULTS_DIR / "resnet18" / "resnet18_oof.npz",
        "familiarity": CORRECTED_DIR / "resnet18_corrected.npz",
    },
    "EfficientNet-B0": {
        "oof": RESULTS_DIR / "efficientnet_b0" / "efficientnet_b0_oof.npz",
        "familiarity": CORRECTED_DIR / "efficientnet_b0_corrected.npz",
    },
}

SPLIT_SEEDS = [20261109, 20262109]
TARGET_CALIBRATION_RECALL = 0.90
BOOTSTRAP_REPLICATES = 3000
BOOTSTRAP_SEED = 20261110

METHOD_FEATURES = {
    "C": ["C"],
    "H": ["H"],
    "S": ["S"],
    "C+H": ["C", "H"],
    "C+S": ["C", "S"],
    "H+S": ["H", "S"],
    "C+H+S": ["C", "H", "S"],
}

FEATURE_INDEX = {"C": 0, "H": 1, "S": 2}
METHODS = list(METHOD_FEATURES.keys())


# ------------------------------------------------
# 2. Metric functions
# ------------------------------------------------

def calculate_metrics(y_true, flagged, weights=None):
    """
    flagged = 1 means the system reflects/flags the prediction.
    y_true = 1 means the classifier prediction is incorrect.
    """

    y_true = np.asarray(y_true).astype(int)
    flagged = np.asarray(flagged).astype(int)

    if weights is None:
        weights = np.ones(len(y_true), dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    tp = np.sum(weights * ((y_true == 1) & (flagged == 1)))
    fp = np.sum(weights * ((y_true == 0) & (flagged == 1)))
    fn = np.sum(weights * ((y_true == 1) & (flagged == 0)))
    tn = np.sum(weights * ((y_true == 0) & (flagged == 0)))

    total = tp + fp + fn + tn
    flagged_total = tp + fp
    accepted_total = fn + tn
    actual_errors = tp + fn
    actual_correct = fp + tn

    return {
        "Recall": tp / actual_errors if actual_errors else np.nan,
        "Precision": tp / flagged_total if flagged_total else np.nan,
        "False_reflection_rate": (
            fp / actual_correct if actual_correct else np.nan
        ),
        "Coverage": accepted_total / total if total else np.nan,
        "Accepted_risk": (
            fn / accepted_total if accepted_total else np.nan
        ),
    }


def select_error_threshold(scores, errors, target_recall=0.90):
    """
    Select the threshold using calibration errors only.
    A sample is flagged when score >= threshold.
    """

    scores = np.asarray(scores, dtype=float)
    errors = np.asarray(errors).astype(bool)

    error_scores = scores[errors]

    if len(error_scores) == 0:
        raise ValueError("Calibration partition contains no errors.")

    k = int(np.ceil(target_recall * len(error_scores)))
    k = max(1, min(k, len(error_scores)))

    threshold = np.sort(error_scores)[-k]

    calibration_flags = scores >= threshold
    actual_recall = np.sum(calibration_flags & errors) / np.sum(errors)

    return float(threshold), float(actual_recall)


# ------------------------------------------------
# 3. Load and validate the three OOF datasets
# ------------------------------------------------

architecture_data = {}

for architecture, paths in ARCHITECTURES.items():

    print("\n" + "=" * 75)
    print("LOADING:", architecture)
    print("=" * 75)

    assert paths["oof"].exists(), f"Missing OOF file: {paths['oof']}"
    assert paths["familiarity"].exists(), (
        f"Missing familiarity file: {paths['familiarity']}"
    )

    with np.load(paths["oof"], allow_pickle=True) as oof:
        logits_key = "oof_logits" if "oof_logits" in oof.files else "logits"
        labels_key = "oof_labels" if "oof_labels" in oof.files else "labels"
        predictions_key = (
            "oof_predictions"
            if "oof_predictions" in oof.files
            else "predictions"
        )
        folds_key = "oof_folds" if "oof_folds" in oof.files else "folds"

        logits = oof[logits_key].astype(np.float64)
        labels = oof[labels_key].astype(int)
        predictions = oof[predictions_key].astype(int)
        folds = oof[folds_key].astype(int)
        class_names = oof["class_names"].astype(str)

    with np.load(paths["familiarity"], allow_pickle=True) as fam:
        familiarity = fam["familiarity"].astype(np.float64)
        low_familiarity = fam["low_familiarity"].astype(np.float64)

        fam_labels = fam["labels"].astype(int)
        fam_predictions = fam["predictions"].astype(int)
        fam_folds = fam["folds"].astype(int)
        fam_errors = fam["errors"].astype(int)
        fam_class_names = fam["class_names"].astype(str)

    # Strong integrity checks before using the signals.
    assert logits.shape == (20638, 15), logits.shape
    assert labels.shape == predictions.shape == folds.shape == (20638,)
    assert familiarity.shape == low_familiarity.shape == (20638,)

    np.testing.assert_array_equal(labels, fam_labels)
    np.testing.assert_array_equal(predictions, fam_predictions)
    np.testing.assert_array_equal(folds, fam_folds)
    np.testing.assert_array_equal(class_names, fam_class_names)
    np.testing.assert_array_equal(
        (predictions != labels).astype(int),
        fam_errors
    )
    np.testing.assert_allclose(
        low_familiarity,
        1.0 - familiarity,
        rtol=1e-7,
        atol=1e-7
    )

    # Reconstruct C and H from the original OOF logits.
    shifted_logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted_logits)
    probabilities = exp_logits / exp_logits.sum(axis=1, keepdims=True)

    low_confidence = 1.0 - probabilities.max(axis=1)

    entropy = -np.sum(
        probabilities * np.log(np.clip(probabilities, 1e-12, 1.0)),
        axis=1
    )
    normalized_entropy = entropy / np.log(logits.shape[1])

    # Feature order: C, H, S.
    X = np.column_stack([
        low_confidence,
        normalized_entropy,
        low_familiarity
    ])

    errors = (predictions != labels).astype(int)

    assert np.isfinite(X).all(), f"Non-finite signal in {architecture}"

    architecture_data[architecture] = {
        "X": X,
        "labels": labels,
        "predictions": predictions,
        "folds": folds,
        "errors": errors,
        "class_names": class_names,
    }

    print(
        f"{architecture}: {len(labels)} samples | "
        f"{errors.sum()} errors | integrity PASS"
    )


# ------------------------------------------------
# 4. Run repeated fold-wise meta-learning evaluation
# ------------------------------------------------

fold_result_rows = []
test_prediction_rows = []
calibration_rows = []
split_manifest_rows = []

for architecture, data in architecture_data.items():

    X = data["X"]
    errors = data["errors"]
    folds = data["folds"]

    print("\n" + "=" * 80)
    print("EVALUATING:", architecture)
    print("=" * 80)

    for fold in sorted(np.unique(folds)):

        fold_indices = np.flatnonzero(folds == fold)
        fold_errors = errors[fold_indices]

        for seed in SPLIT_SEEDS:

            # First split: 50% meta-training, 50% remainder.
            meta_train_idx, remainder_idx = train_test_split(
                fold_indices,
                test_size=0.50,
                random_state=seed,
                stratify=fold_errors
            )

            # Second split: divide remainder equally into calibration/test.
            cal_idx, test_idx = train_test_split(
                remainder_idx,
                test_size=0.50,
                random_state=seed,
                stratify=errors[remainder_idx]
            )

            assert len(meta_train_idx) + len(cal_idx) + len(test_idx) == len(fold_indices)
            assert len(set(meta_train_idx) & set(cal_idx)) == 0
            assert len(set(meta_train_idx) & set(test_idx)) == 0
            assert len(set(cal_idx) & set(test_idx)) == 0

            print(
                f"\nFold {fold}, seed {seed}: "
                f"meta-train={len(meta_train_idx)}, "
                f"calibration={len(cal_idx)}, "
                f"test={len(test_idx)}"
            )

            split_manifest_rows.append({
                "Architecture": architecture,
                "Fold": int(fold),
                "Split_seed": int(seed),
                "Meta_train_n": len(meta_train_idx),
                "Calibration_n": len(cal_idx),
                "Test_n": len(test_idx),
                "Meta_train_errors": int(errors[meta_train_idx].sum()),
                "Calibration_errors": int(errors[cal_idx].sum()),
                "Test_errors": int(errors[test_idx].sum()),
            })

            # Baseline: direct low-confidence score.
            baseline_cal_scores = X[cal_idx, FEATURE_INDEX["C"]]
            baseline_test_scores = X[test_idx, FEATURE_INDEX["C"]]

            baseline_threshold, baseline_cal_recall = select_error_threshold(
                baseline_cal_scores,
                errors[cal_idx],
                TARGET_CALIBRATION_RECALL
            )

            baseline_test_flags = (
                baseline_test_scores >= baseline_threshold
            ).astype(int)

            baseline_metrics = calculate_metrics(
                errors[test_idx],
                baseline_test_flags
            )

            fold_result_rows.append({
                "Architecture": architecture,
                "Fold": int(fold),
                "Split_seed": int(seed),
                "Method": "Baseline confidence",
                "Threshold": baseline_threshold,
                "Calibration_error_recall": baseline_cal_recall,
                **baseline_metrics
            })

            calibration_rows.append({
                "Architecture": architecture,
                "Fold": int(fold),
                "Split_seed": int(seed),
                "Method": "Baseline confidence",
                "Calibration_errors": int(errors[cal_idx].sum()),
                "Threshold": baseline_threshold,
                "Calibration_error_recall": baseline_cal_recall,
            })

            test_prediction_rows.append(pd.DataFrame({
                "Architecture": architecture,
                "Fold": int(fold),
                "Split_seed": int(seed),
                "Image_index": test_idx,
                "Method": "Baseline confidence",
                "True_error": errors[test_idx],
                "Risk_score": baseline_test_scores,
                "Threshold": baseline_threshold,
                "Flagged": baseline_test_flags,
            }))

            # Fit each reflection method using meta-training only.
            for method, feature_names in METHOD_FEATURES.items():

                feature_columns = [
                    FEATURE_INDEX[name] for name in feature_names
                ]

                X_train = X[meta_train_idx][:, feature_columns]
                X_cal = X[cal_idx][:, feature_columns]
                X_test = X[test_idx][:, feature_columns]

                model = Pipeline([
                    ("scaler", StandardScaler()),
                    ("logistic", LogisticRegression(
                        max_iter=5000,
                        solver="lbfgs",
                        random_state=0
                    ))
                ])

                model.fit(X_train, errors[meta_train_idx])

                error_class_position = list(model.classes_).index(1)

                cal_scores = model.predict_proba(X_cal)[
                    :, error_class_position
                ]
                test_scores = model.predict_proba(X_test)[
                    :, error_class_position
                ]

                threshold, calibration_recall = select_error_threshold(
                    cal_scores,
                    errors[cal_idx],
                    TARGET_CALIBRATION_RECALL
                )

                test_flags = (test_scores >= threshold).astype(int)

                metrics = calculate_metrics(
                    errors[test_idx],
                    test_flags
                )

                fold_result_rows.append({
                    "Architecture": architecture,
                    "Fold": int(fold),
                    "Split_seed": int(seed),
                    "Method": method,
                    "Threshold": threshold,
                    "Calibration_error_recall": calibration_recall,
                    **metrics
                })

                calibration_rows.append({
                    "Architecture": architecture,
                    "Fold": int(fold),
                    "Split_seed": int(seed),
                    "Method": method,
                    "Calibration_errors": int(errors[cal_idx].sum()),
                    "Threshold": threshold,
                    "Calibration_error_recall": calibration_recall,
                })

                test_prediction_rows.append(pd.DataFrame({
                    "Architecture": architecture,
                    "Fold": int(fold),
                    "Split_seed": int(seed),
                    "Image_index": test_idx,
                    "Method": method,
                    "True_error": errors[test_idx],
                    "Risk_score": test_scores,
                    "Threshold": threshold,
                    "Flagged": test_flags,
                }))


# ------------------------------------------------
# 5. Assemble and save per-fold and per-test outputs
# ------------------------------------------------

fold_results = pd.DataFrame(fold_result_rows)
test_predictions = pd.concat(test_prediction_rows, ignore_index=True)
calibration_check = pd.DataFrame(calibration_rows)
split_manifest = pd.DataFrame(split_manifest_rows)

fold_results_path = OUTPUT_DIR / "evaluation_v3_corrected_fold_results.csv"
test_predictions_path = OUTPUT_DIR / "evaluation_v3_corrected_test_predictions.csv"
calibration_path = OUTPUT_DIR / "evaluation_v3_corrected_calibration_check.csv"
split_manifest_path = OUTPUT_DIR / "evaluation_v3_corrected_split_manifest.csv"

fold_results.to_csv(fold_results_path, index=False)
test_predictions.to_csv(test_predictions_path, index=False)
calibration_check.to_csv(calibration_path, index=False)
split_manifest.to_csv(split_manifest_path, index=False)


# ------------------------------------------------
# 6. Verify calibration target
# ------------------------------------------------

print("\n" + "=" * 90)
print("CALIBRATION RECALL CHECK")
print("=" * 90)

calibration_summary = (
    calibration_check
    .groupby(["Architecture", "Method"])["Calibration_error_recall"]
    .agg(["min", "mean", "max"])
    .reset_index()
)

print(calibration_summary.to_string(index=False))

assert (
    calibration_check["Calibration_error_recall"].min()
    >= TARGET_CALIBRATION_RECALL - 1e-12
), "At least one calibration partition failed the target recall."


# ------------------------------------------------
# 7. Pooled held-out test metrics
# ------------------------------------------------

pooled_rows = []

for architecture in ARCHITECTURES:

    arch_df = test_predictions[
        test_predictions["Architecture"] == architecture
    ]

    for method in ["Baseline confidence"] + METHODS:

        method_df = arch_df[arch_df["Method"] == method]

        metrics = calculate_metrics(
            method_df["True_error"].to_numpy(),
            method_df["Flagged"].to_numpy()
        )

        pooled_rows.append({
            "Architecture": architecture,
            "Method": method,
            "Test_samples": len(method_df),
            "Test_errors": int(method_df["True_error"].sum()),
            **metrics
        })

pooled_results = pd.DataFrame(pooled_rows)
pooled_path = OUTPUT_DIR / "evaluation_v3_corrected_pooled_results.csv"
pooled_results.to_csv(pooled_path, index=False)

print("\n" + "=" * 90)
print("POOLED HELD-OUT TEST RESULTS")
print("=" * 90)
print(
    pooled_results.round(5).to_string(index=False)
)

print("\nCorrected held-out predictions and pooled metrics are ready.")
print("Evaluation directory:", OUTPUT_DIR)

### 8.1 Paired bootstrap for threshold-based evaluation

This cell compares each reflection method with the confidence baseline using a paired, fold-stratified, image-cluster bootstrap. The architecture is included in the case key so identical image indices across different architectures are not incorrectly treated as duplicate records.

In [ ]:
# ================================================================
# PBL-3: RESUME FROM SECTION 8 — PAIRED BOOTSTRAP
# ================================================================

import numpy as np
import pandas as pd
import json
import zipfile

# Include Architecture in the key.
CASE_COLUMNS = [
    "Architecture",
    "Fold",
    "Split_seed",
    "Image_index"
]

# Confirm one row per image/split/method.
duplicate_count = test_predictions.duplicated(
    CASE_COLUMNS + ["Method"]
).sum()

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate image/split/method rows."
)

# Pivot method decisions into columns.
wide_flags = test_predictions.pivot(
    index=CASE_COLUMNS,
    columns="Method",
    values="Flagged"
)

case_metadata = (
    test_predictions
    .drop_duplicates(CASE_COLUMNS)
    .set_index(CASE_COLUMNS)[["True_error"]]
)

wide_cases = case_metadata.join(wide_flags).reset_index()

assert len(wide_cases) == len(test_predictions) // 8
assert wide_cases["Architecture"].notna().all()

print("Bootstrap case table:", wide_cases.shape)
print("Duplicate case rows:", wide_cases.duplicated(CASE_COLUMNS).sum())


# ------------------------------------------------
# Paired, fold-stratified, image-cluster bootstrap
# ------------------------------------------------

METRICS = [
    "Recall",
    "Precision",
    "False_reflection_rate",
    "Coverage",
    "Accepted_risk",
]

HIGHER_IS_BETTER = {
    "Recall": True,
    "Precision": True,
    "False_reflection_rate": False,
    "Coverage": True,
    "Accepted_risk": False,
}

bootstrap_rng = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows = []

for architecture in ARCHITECTURES:

    arch_cases = (
        wide_cases[
            wide_cases["Architecture"] == architecture
        ]
        .reset_index(drop=True)
    )

    y = arch_cases["True_error"].to_numpy(dtype=int)
    folds_for_cases = arch_cases["Fold"].to_numpy(dtype=int)
    image_indices = arch_cases["Image_index"].to_numpy(dtype=int)

    baseline_flags = arch_cases[
        "Baseline confidence"
    ].to_numpy(dtype=int)

    bootstrap_differences = {
        method: {
            metric: np.empty(BOOTSTRAP_REPLICATES)
            for metric in METRICS
        }
        for method in METHODS
    }

    fold_positions = {
        fold: np.flatnonzero(folds_for_cases == fold)
        for fold in sorted(np.unique(folds_for_cases))
    }

    for b in range(BOOTSTRAP_REPLICATES):

        weights = np.zeros(len(arch_cases), dtype=float)

        for fold, positions in fold_positions.items():

            # Resample original image IDs within each outer fold.
            # An image appearing in both split seeds receives the
            # same bootstrap multiplicity in both observations.
            unique_images = np.unique(image_indices[positions])

            sampled_images = bootstrap_rng.choice(
                unique_images,
                size=len(unique_images),
                replace=True
            )

            image_counts = (
                pd.Series(sampled_images)
                .value_counts()
                .to_dict()
            )

            weights[positions] = np.array([
                image_counts.get(image_id, 0)
                for image_id in image_indices[positions]
            ], dtype=float)

        baseline_metrics = calculate_metrics(
            y,
            baseline_flags,
            weights=weights
        )

        for method in METHODS:

            method_flags = arch_cases[
                method
            ].to_numpy(dtype=int)

            method_metrics = calculate_metrics(
                y,
                method_flags,
                weights=weights
            )

            for metric in METRICS:
                bootstrap_differences[method][metric][b] = (
                    method_metrics[metric]
                    - baseline_metrics[metric]
                )

    # Compare observed pooled metrics with the baseline.
    for method in METHODS:

        method_pooled = pooled_results[
            (pooled_results["Architecture"] == architecture)
            & (pooled_results["Method"] == method)
        ].iloc[0]

        baseline_pooled = pooled_results[
            (pooled_results["Architecture"] == architecture)
            & (pooled_results["Method"] == "Baseline confidence")
        ].iloc[0]

        for metric in METRICS:

            difference = (
                method_pooled[metric] - baseline_pooled[metric]
            )

            boot_values = bootstrap_differences[method][metric]

            ci_lower, ci_upper = np.percentile(
                boot_values,
                [2.5, 97.5]
            )

            if HIGHER_IS_BETTER[metric]:
                direction_supported = ci_lower > 0
            else:
                direction_supported = ci_upper < 0

            bootstrap_rows.append({
                "Architecture": architecture,
                "Method": method,
                "Metric": metric,
                "Baseline": baseline_pooled[metric],
                "Reflection": method_pooled[metric],
                "Difference": difference,
                "CI_95_lower": ci_lower,
                "CI_95_upper": ci_upper,
                "CI_excludes_zero": (
                    ci_lower > 0 or ci_upper < 0
                ),
                "Improvement_direction_supported": direction_supported,
                "Bootstrap_replicates": BOOTSTRAP_REPLICATES,
            })

    print(f"Completed bootstrap: {architecture}")


# ------------------------------------------------
# Save bootstrap results
# ------------------------------------------------

bootstrap_results = pd.DataFrame(bootstrap_rows)

bootstrap_path = (
    OUTPUT_DIR / "evaluation_v3_corrected_paired_bootstrap.csv"
)

bootstrap_results.to_csv(bootstrap_path, index=False)

print("\n" + "=" * 100)
print("CORRECTED PAIRED BOOTSTRAP RESULTS")
print("=" * 100)

print(
    bootstrap_results.round(5).to_string(index=False)
)


# ------------------------------------------------
# Save manifest
# ------------------------------------------------

manifest = {
    "experiment": "PBL-3 corrected reflection evaluation v3",
    "dataset_size": 20638,
    "number_of_classes": 15,
    "architectures": list(ARCHITECTURES.keys()),
    "signals": {
        "C": "1 - maximum softmax probability",
        "H": "Predictive entropy normalized by log(number of classes)",
        "S": "1 - cosine similarity to predicted-class prototype",
    },
    "familiarity_source": "Corrected fold-consistent familiarity extraction",
    "familiarity_validation": (
        "Zero prediction mismatches for all architectures and folds"
    ),
    "baseline": "Low confidence (1 - maximum softmax probability)",
    "meta_model": "StandardScaler + LogisticRegression",
    "split_seeds": SPLIT_SEEDS,
    "split_proportions": {
        "meta_train": 0.50,
        "calibration": 0.25,
        "test": 0.25,
    },
    "split_method": (
        "Stratified train_test_split within each OOF fold; "
        "first split creates meta-training and remainder; "
        "second split divides remainder into calibration and test."
    ),
    "target_calibration_error_recall": TARGET_CALIBRATION_RECALL,
    "threshold_rule": (
        "Threshold selected from calibration error scores only; "
        "test samples flagged when risk score >= threshold."
    ),
    "bootstrap": {
        "replicates": BOOTSTRAP_REPLICATES,
        "seed": BOOTSTRAP_SEED,
        "method": (
            "Paired fold-stratified image-cluster percentile bootstrap"
        ),
        "confidence_interval": "95% unadjusted percentile interval",
    },
    "cnn_retraining": False,
}

manifest_path = OUTPUT_DIR / "evaluation_v3_corrected_manifest.json"

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)


# ------------------------------------------------
# Create final evaluation bundle
# ------------------------------------------------

files_to_bundle = [
    fold_results_path,
    test_predictions_path,
    calibration_path,
    split_manifest_path,
    pooled_path,
    bootstrap_path,
    manifest_path,
]

bundle_path = RESULTS_DIR / "evaluation_v3_corrected_bundle.zip"

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in files_to_bundle:
        assert path.exists(), f"Missing output file: {path}"
        zf.write(path, arcname=path.name)


# ------------------------------------------------
# Final checks
# ------------------------------------------------

assert len(bootstrap_results) == (
    len(ARCHITECTURES) * len(METHODS) * len(METRICS)
)

assert len(pooled_results) == (
    len(ARCHITECTURES) * (1 + len(METHODS))
)

print("\n" + "=" * 90)
print("CORRECTED EVALUATION V3 COMPLETE")
print("=" * 90)

print("Bootstrap results:", bootstrap_path)
print("Manifest:", manifest_path)
print("Final bundle:", bundle_path)

## 9. Risk-coverage analysis and AURC

Risk-coverage analysis orders predictions from lowest to highest estimated error risk and measures the error rate among accepted predictions as coverage changes. Lower AURC indicates better selective prediction. The paired bootstrap uses 10,000 replicates and adjusts for the 18 method-versus-baseline comparisons.

In [ ]:
# ================================================================
# PBL-3: CORRECTED TEST-SET RISK-COVERAGE AND AURC
# Uses risk scores from corrected evaluation v3.
# ================================================================

from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd


# ------------------------------------------------
# 1. Paths and configuration
# ------------------------------------------------

ROOT = PROJECT_DIR
RESULTS = ROOT / "results"
EVAL_DIR = RESULTS / "evaluation_v3_corrected"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_FILE = (
    EVAL_DIR / "evaluation_v3_corrected_test_predictions.csv"
)

assert PREDICTIONS_FILE.exists(), (
    f"Missing corrected test predictions: {PREDICTIONS_FILE}"
)

ARCHITECTURES = [
    "MobileNetV3-Small",
    "ResNet18",
    "EfficientNet-B0",
]

METHODS = [
    "Baseline confidence",
    "C",
    "H",
    "S",
    "C+H",
    "C+S",
    "H+S",
    "C+H+S",
]

# Six non-baseline methods x three architectures = 18 comparisons.
BOOTSTRAP_METHODS = [
    "H", "S", "C+H", "C+S", "H+S", "C+H+S"
]

SPLIT_SEEDS = [20261109, 20262109]
FOLDS = [1, 2, 3]

BOOTSTRAP_REPLICATES = 10000
BOOTSTRAP_SEED = 20261111
FAMILY_COMPARISONS = 18


# ------------------------------------------------
# 2. Load and validate held-out predictions
# ------------------------------------------------

pred = pd.read_csv(PREDICTIONS_FILE)

required_columns = {
    "Architecture",
    "Fold",
    "Split_seed",
    "Image_index",
    "Method",
    "True_error",
    "Risk_score",
}

missing = required_columns - set(pred.columns)
assert not missing, f"Missing columns: {missing}"

CASE_COLUMNS = [
    "Architecture",
    "Fold",
    "Split_seed",
    "Image_index",
]

assert set(pred["Architecture"].unique()) == set(ARCHITECTURES)
assert set(pred["Method"].unique()) == set(METHODS)
assert set(pred["Split_seed"].unique()) == set(SPLIT_SEEDS)
assert set(pred["Fold"].unique()) == set(FOLDS)

assert pred["Risk_score"].notna().all()
assert np.isfinite(pred["Risk_score"]).all()
assert pred["True_error"].isin([0, 1]).all()

duplicates = pred.duplicated(CASE_COLUMNS + ["Method"]).sum()
assert duplicates == 0, f"Duplicate prediction rows: {duplicates}"

# Every case must have the same ground-truth error label for all methods.
label_counts = (
    pred.groupby(CASE_COLUMNS)["True_error"]
    .nunique()
)
assert label_counts.max() == 1

# One row per image/fold/seed, with one risk score per method.
wide_scores = pred.pivot(
    index=CASE_COLUMNS,
    columns="Method",
    values="Risk_score",
).reset_index()

wide_labels = (
    pred.drop_duplicates(CASE_COLUMNS)
    [CASE_COLUMNS + ["True_error"]]
)

wide = wide_labels.merge(
    wide_scores,
    on=CASE_COLUMNS,
    validate="one_to_one",
)

assert len(wide) == 3 * 3 * 2 * 1720
assert not wide.duplicated(CASE_COLUMNS).any()

partition_sizes = (
    wide.groupby(["Architecture", "Fold", "Split_seed"])
    .size()
)
assert (partition_sizes == 1720).all()

print("Loaded held-out cases:", len(wide))
print("Duplicate cases:", wide.duplicated(CASE_COLUMNS).sum())
print("Partition-size check: PASS")


# ------------------------------------------------
# 3. Risk-coverage calculation
# ------------------------------------------------

COVERAGE_LEVELS = [0.50, 0.70, 0.80, 0.90, 0.95, 1.00]


def calculate_risk_coverage(y, scores):
    """
    Accept samples in ascending predicted error-risk order.

    Coverage = fraction of samples accepted.
    Selective risk = error fraction among accepted samples.
    AURC = trapezoidal area under the risk-coverage curve.
    """

    y = np.asarray(y, dtype=int)
    scores = np.asarray(scores, dtype=float)

    order = np.argsort(scores, kind="mergesort")
    sorted_errors = y[order]

    n = len(y)
    accepted = np.arange(1, n + 1)

    coverage = accepted / n
    cumulative_errors = np.cumsum(sorted_errors)
    risk = cumulative_errors / accepted

    aurc = float(np.trapezoid(risk, coverage))

    risk_at_coverage = {
        f"Risk_at_{int(level * 100)}pct": float(
            np.interp(level, coverage, risk)
        )
        for level in COVERAGE_LEVELS
    }

    return {
        "order": order,
        "coverage": coverage,
        "risk": risk,
        "AURC": aurc,
        "Error_prevalence": float(y.mean()),
        **risk_at_coverage,
    }


# ------------------------------------------------
# 4. Compute curves and fold/seed-level AURC
# ------------------------------------------------

curve_frames = []
foldwise_rows = []

for architecture in ARCHITECTURES:

    arch_data = wide[
        wide["Architecture"] == architecture
    ]

    for fold in FOLDS:
        for seed in SPLIT_SEEDS:

            part = (
                arch_data[
                    (arch_data["Fold"] == fold)
                    & (arch_data["Split_seed"] == seed)
                ]
                .sort_values("Image_index")
                .reset_index(drop=True)
            )

            y = part["True_error"].to_numpy(dtype=int)

            assert len(part) == 1720

            for method in METHODS:

                result = calculate_risk_coverage(
                    y,
                    part[method].to_numpy(dtype=float)
                )

                foldwise_rows.append({
                    "Architecture": architecture,
                    "Fold": fold,
                    "Split_seed": seed,
                    "Method": method,
                    "Samples": len(y),
                    "Errors": int(y.sum()),
                    "AURC": result["AURC"],
                    "Error_prevalence": result["Error_prevalence"],
                    **{
                        k: v for k, v in result.items()
                        if k.startswith("Risk_at_")
                    }
                })

                curve_frames.append(pd.DataFrame({
                    "Architecture": architecture,
                    "Fold": fold,
                    "Split_seed": seed,
                    "Method": method,
                    "Rank": np.arange(1, len(y) + 1),
                    "Coverage": result["coverage"],
                    "Risk": result["risk"],
                }))


foldwise = pd.DataFrame(foldwise_rows)
curves = pd.concat(curve_frames, ignore_index=True)


# ------------------------------------------------
# 5. Summarize the six held-out partitions
# ------------------------------------------------

summary = (
    foldwise
    .groupby(["Architecture", "Method"])
    .agg(
        AURC_mean=("AURC", "mean"),
        AURC_std=("AURC", "std"),
        Error_prevalence_mean=("Error_prevalence", "mean"),
        Risk_at_50pct_mean=("Risk_at_50pct", "mean"),
        Risk_at_70pct_mean=("Risk_at_70pct", "mean"),
        Risk_at_80pct_mean=("Risk_at_80pct", "mean"),
        Risk_at_90pct_mean=("Risk_at_90pct", "mean"),
        Risk_at_95pct_mean=("Risk_at_95pct", "mean"),
        Risk_at_100pct_mean=("Risk_at_100pct", "mean"),
    )
    .reset_index()
)

print("\n" + "=" * 110)
print("CORRECTED RISK-COVERAGE SUMMARY — LOWER AURC IS BETTER")
print("=" * 110)
print(summary.round(6).to_string(index=False))


# ------------------------------------------------
# 6. Bootstrap helper for weighted AURC
# ------------------------------------------------

def weighted_aurc(sorted_errors, sorted_weights):
    """
    Bootstrap-weighted AURC using a fixed, previously sorted ranking.
    This is equivalent to resampling image IDs with replacement,
    then retaining the same risk-score ordering.
    """

    sorted_errors = np.asarray(sorted_errors, dtype=float)
    sorted_weights = np.asarray(sorted_weights, dtype=float)

    cumulative_weight = np.cumsum(sorted_weights)
    total_weight = cumulative_weight[-1]

    if total_weight <= 0:
        return np.nan

    cumulative_errors = np.cumsum(
        sorted_errors * sorted_weights
    )

    valid = cumulative_weight > 0

    coverage = cumulative_weight[valid] / total_weight
    risk = (
        cumulative_errors[valid]
        / cumulative_weight[valid]
    )

    return float(np.trapezoid(risk, coverage))


# ------------------------------------------------
# 7. Prepare sorted arrays and image-cluster mappings
# ------------------------------------------------

# For each architecture/fold, bootstrap image IDs shared across
# the two split seeds. This preserves pairing across seeds.

partition_cache = {}
fold_image_maps = {}

for architecture in ARCHITECTURES:

    arch_data = wide[
        wide["Architecture"] == architecture
    ]

    for fold in FOLDS:

        fold_data = arch_data[
            arch_data["Fold"] == fold
        ]

        unique_images, inverse = np.unique(
            fold_data["Image_index"].to_numpy(dtype=int),
            return_inverse=True
        )

        fold_image_maps[(architecture, fold)] = {
            "unique_images": unique_images,
            "inverse_by_case": inverse,
            "case_keys": list(
                zip(
                    fold_data["Split_seed"].to_numpy(dtype=int),
                    fold_data["Image_index"].to_numpy(dtype=int)
                )
            )
        }

        # Map each fold/seed partition to its inverse image indices.
        for seed in SPLIT_SEEDS:

            part = (
                fold_data[
                    fold_data["Split_seed"] == seed
                ]
                .sort_values("Image_index")
                .reset_index(drop=True)
            )

            y = part["True_error"].to_numpy(dtype=int)
            image_ids = part["Image_index"].to_numpy(dtype=int)

            _, inverse_part = np.unique(
                unique_images,
                return_inverse=True
            )

            image_to_position = {
                image_id: pos
                for pos, image_id in enumerate(unique_images)
            }

            inverse_rows = np.array([
                image_to_position[x] for x in image_ids
            ], dtype=int)

            for method in ["Baseline confidence"] + BOOTSTRAP_METHODS:

                scores = part[method].to_numpy(dtype=float)
                order = np.argsort(scores, kind="mergesort")

                partition_cache[
                    (architecture, fold, seed, method)
                ] = {
                    "sorted_errors": y[order],
                    "sorted_inverse_images": inverse_rows[order],
                }


# ------------------------------------------------
# 8. Paired, fold-stratified image-cluster bootstrap
# ------------------------------------------------

rng = np.random.default_rng(BOOTSTRAP_SEED)

bootstrap_differences = {
    (architecture, method): np.empty(BOOTSTRAP_REPLICATES)
    for architecture in ARCHITECTURES
    for method in BOOTSTRAP_METHODS
}

for architecture in ARCHITECTURES:

    print(f"\nBootstrapping {architecture}...")

    for b in range(BOOTSTRAP_REPLICATES):

        replicate_auc = {
            "Baseline confidence": []
        }
        for method in BOOTSTRAP_METHODS:
            replicate_auc[method] = []

        for fold in FOLDS:

            image_map = fold_image_maps[(architecture, fold)]
            unique_images = image_map["unique_images"]
            n_unique = len(unique_images)

            # Sample original image IDs once per fold.
            # The same sampled multiplicities are used for both seeds.
            sampled_positions = rng.integers(
                0,
                n_unique,
                size=n_unique
            )

            counts = np.bincount(
                sampled_positions,
                minlength=n_unique
            )

            for seed in SPLIT_SEEDS:

                for method in ["Baseline confidence"] + BOOTSTRAP_METHODS:

                    cached = partition_cache[
                        (architecture, fold, seed, method)
                    ]

                    weights = counts[
                        cached["sorted_inverse_images"]
                    ]

                    auc = weighted_aurc(
                        cached["sorted_errors"],
                        weights
                    )

                    replicate_auc[method].append(auc)

        baseline_mean = np.nanmean(
            replicate_auc["Baseline confidence"]
        )

        for method in BOOTSTRAP_METHODS:

            method_mean = np.nanmean(replicate_auc[method])

            bootstrap_differences[
                (architecture, method)
            ][b] = method_mean - baseline_mean

    print(
        f"Completed {BOOTSTRAP_REPLICATES:,} paired replicates."
    )


# ------------------------------------------------
# 9. Bootstrap confidence intervals and correction
# ------------------------------------------------

bootstrap_rows = []

alpha_bonferroni = 0.05 / FAMILY_COMPARISONS
lower_q = alpha_bonferroni / 2
upper_q = 1 - alpha_bonferroni / 2

for architecture in ARCHITECTURES:

    baseline_auc = (
        foldwise[
            (foldwise["Architecture"] == architecture)
            & (foldwise["Method"] == "Baseline confidence")
        ]["AURC"].mean()
    )

    for method in BOOTSTRAP_METHODS:

        method_auc = (
            foldwise[
                (foldwise["Architecture"] == architecture)
                & (foldwise["Method"] == method)
            ]["AURC"].mean()
        )

        observed_difference = method_auc - baseline_auc
        boot = bootstrap_differences[(architecture, method)]

        ci95_lower, ci95_upper = np.quantile(
            boot,
            [0.025, 0.975]
        )

        bonf_lower, bonf_upper = np.quantile(
            boot,
            [lower_q, upper_q]
        )

        bootstrap_rows.append({
            "Architecture": architecture,
            "Method": method,
            "Mean_fold_baseline_AURC": baseline_auc,
            "Mean_fold_method_AURC": method_auc,
            "Mean_AURC_difference": observed_difference,
            "CI_95_lower": ci95_lower,
            "CI_95_upper": ci95_upper,
            "Lower_AURC_supported_unadjusted": ci95_upper < 0,
            "Bonferroni_CI_lower": bonf_lower,
            "Bonferroni_CI_upper": bonf_upper,
            "Lower_AURC_supported_Bonferroni": bonf_upper < 0,
            "Bonferroni_comparisons": FAMILY_COMPARISONS,
            "Bootstrap_replicates": BOOTSTRAP_REPLICATES,
        })

bootstrap_results = pd.DataFrame(bootstrap_rows)

print("\n" + "=" * 110)
print("PAIRED AURC BOOTSTRAP — NEGATIVE DIFFERENCE FAVOURS REFLECTION")
print("=" * 110)
print(
    bootstrap_results.round(7).to_string(index=False)
)


# ------------------------------------------------
# 10. Save outputs
# ------------------------------------------------

foldwise_path = (
    EVAL_DIR / "evaluation_v3_corrected_risk_coverage_foldwise.csv"
)

summary_path = (
    EVAL_DIR / "evaluation_v3_corrected_risk_coverage_summary.csv"
)

curves_path = (
    EVAL_DIR / "evaluation_v3_corrected_risk_coverage_curves.csv"
)

bootstrap_path = (
    EVAL_DIR / "evaluation_v3_corrected_aurc_bootstrap.csv"
)

foldwise.to_csv(foldwise_path, index=False)
summary.to_csv(summary_path, index=False)
curves.to_csv(curves_path, index=False)
bootstrap_results.to_csv(bootstrap_path, index=False)


# ------------------------------------------------
# 11. Save reproducibility manifest
# ------------------------------------------------

manifest = {
    "analysis": "Corrected v3 held-out risk-coverage evaluation",
    "input": str(PREDICTIONS_FILE),
    "risk_score_source": (
        "Risk_score from the corrected v3 logistic reflection models; "
        "baseline uses low-confidence risk score."
    ),
    "acceptance_rule": (
        "Samples are accepted in ascending predicted error-risk order."
    ),
    "risk_definition": (
        "Cumulative classification errors divided by accepted samples."
    ),
    "aurc": "Trapezoidal area under the risk-coverage curve.",
    "partitions": {
        "folds": FOLDS,
        "split_seeds": SPLIT_SEEDS,
        "test_samples_per_partition": 1720,
        "partitions_per_architecture": 6,
    },
    "bootstrap": {
        "replicates": BOOTSTRAP_REPLICATES,
        "seed": BOOTSTRAP_SEED,
        "sampling": (
            "Paired image-cluster bootstrap, stratified by OOF fold; "
            "image multiplicities shared across split seeds."
        ),
        "unadjusted_interval": "95% percentile interval",
        "bonferroni_family_size": FAMILY_COMPARISONS,
        "bonferroni_interval": (
            f"{100 * (1 - 0.05 / FAMILY_COMPARISONS):.4f}% "
            "two-sided family-wise percentile interval"
        ),
    },
    "cnn_retraining": False,
}

manifest_path = (
    EVAL_DIR / "evaluation_v3_corrected_risk_coverage_manifest.json"
)

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)


# ------------------------------------------------
# 12. Create complete evidence bundle
# ------------------------------------------------

bundle_files = [
    EVAL_DIR / "evaluation_v3_corrected_fold_results.csv",
    EVAL_DIR / "evaluation_v3_corrected_test_predictions.csv",
    EVAL_DIR / "evaluation_v3_corrected_calibration_check.csv",
    EVAL_DIR / "evaluation_v3_corrected_split_manifest.csv",
    EVAL_DIR / "evaluation_v3_corrected_pooled_results.csv",
    EVAL_DIR / "evaluation_v3_corrected_paired_bootstrap.csv",
    EVAL_DIR / "evaluation_v3_corrected_manifest.json",
    foldwise_path,
    summary_path,
    curves_path,
    bootstrap_path,
    manifest_path,
]

complete_bundle = (
    RESULTS / "evaluation_v3_corrected_complete_bundle.zip"
)

with zipfile.ZipFile(
    complete_bundle,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for path in bundle_files:
        assert path.exists(), f"Missing output: {path}"
        zf.write(path, arcname=path.name)


# ------------------------------------------------
# 13. Final integrity checks
# ------------------------------------------------

assert len(foldwise) == 3 * 6 * 8
assert len(summary) == 3 * 8
assert len(bootstrap_results) == 3 * 6

assert np.isfinite(foldwise["AURC"]).all()
assert np.isfinite(bootstrap_results["Mean_AURC_difference"]).all()

print("\n" + "=" * 90)
print("CORRECTED RISK-COVERAGE ANALYSIS COMPLETE")
print("=" * 90)

print("Foldwise:", foldwise_path)
print("Summary:", summary_path)
print("Curves:", curves_path)
print("AURC bootstrap:", bootstrap_path)
print("Manifest:", manifest_path)
print("Complete evidence bundle:", complete_bundle)

## 10. Error-probability calibration diagnostics

Brier score, negative log-likelihood, and adaptive ECE are calculated for the held-out error-risk scores. These metrics describe both probability quality and calibration; adaptive ECE is the calibration-focused diagnostic. The raw confidence baseline and fitted logistic-regression risk models are distinct scores and should not be conflated.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

EVAL_DIR = PROJECT_DIR / "results" / "evaluation_v3_corrected"

INPUT = EVAL_DIR / "evaluation_v3_corrected_test_predictions.csv"

df = pd.read_csv(INPUT)

METHODS = [
    "Baseline confidence", "C", "H", "S",
    "C+H", "C+S", "H+S", "C+H+S"
]

# ------------------------------------------------------------
# Calibration metric functions
# ------------------------------------------------------------

def calibration_metrics(y, scores, bins=15):

    y = np.asarray(y, dtype=float)
    scores = np.clip(
        np.asarray(scores, dtype=float),
        1e-8, 1 - 1e-8
    )

    brier = np.mean((scores - y) ** 2)

    nll = -np.mean(
        y * np.log(scores) +
        (1 - y) * np.log(1 - scores)
    )

    order = np.argsort(scores)
    partitions = np.array_split(order, bins)

    ece = 0.0
    bin_records = []

    for bin_number, indices in enumerate(partitions, start=1):

        if len(indices) == 0:
            continue

        predicted = float(scores[indices].mean())
        observed = float(y[indices].mean())

        ece += (
            len(indices) / len(y)
            * abs(predicted - observed)
        )

        bin_records.append({
            "Bin": bin_number,
            "Count": len(indices),
            "Mean_predicted_error": predicted,
            "Observed_error_rate": observed
        })

    return {
        "Brier": float(brier),
        "NLL": float(nll),
        "Adaptive_ECE": float(ece)
    }, bin_records


# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

required = {
    "Architecture", "Fold", "Split_seed",
    "Image_index", "Method", "True_error", "Risk_score"
}

assert required.issubset(df.columns)
assert set(df["Method"].unique()) == set(METHODS)
assert df["Risk_score"].between(0, 1).all()

print("Loaded:", len(df), "held-out prediction records")

# ------------------------------------------------------------
# Calculate calibration separately per partition
# ------------------------------------------------------------

partition_results = []
bin_results = []

group_columns = [
    "Architecture", "Fold", "Split_seed", "Method"
]

for keys, group in df.groupby(group_columns):

    architecture, fold, seed, method = keys

    assert len(group) == 1720, (
        f"Unexpected partition size: {keys}, n={len(group)}"
    )

    y = group["True_error"].to_numpy(dtype=int)
    scores = group["Risk_score"].to_numpy(dtype=float)

    metrics, bins = calibration_metrics(y, scores)

    partition_results.append({
        "Architecture": architecture,
        "Fold": int(fold),
        "Split_seed": int(seed),
        "Method": method,
        "Samples": len(group),
        "Error_prevalence": float(y.mean()),
        **metrics
    })

    for record in bins:
        bin_results.append({
            "Architecture": architecture,
            "Fold": int(fold),
            "Split_seed": int(seed),
            "Method": method,
            **record
        })

partition_df = pd.DataFrame(partition_results)
bins_df = pd.DataFrame(bin_results)

# ------------------------------------------------------------
# Macro-average across the six evaluation partitions
# ------------------------------------------------------------

summary = (
    partition_df
    .groupby(["Architecture", "Method"])
    .agg(
        Brier_mean=("Brier", "mean"),
        Brier_std=("Brier", "std"),
        NLL_mean=("NLL", "mean"),
        NLL_std=("NLL", "std"),
        Adaptive_ECE_mean=("Adaptive_ECE", "mean"),
        Adaptive_ECE_std=("Adaptive_ECE", "std"),
        Error_prevalence_mean=("Error_prevalence", "mean")
    )
    .reset_index()
)

# ------------------------------------------------------------
# Compare composites with baseline
# ------------------------------------------------------------

comparison_rows = []

for architecture in df["Architecture"].unique():

    subset = summary[
        summary["Architecture"] == architecture
    ]

    baseline = subset[
        subset["Method"] == "Baseline confidence"
    ].iloc[0]

    for _, row in subset.iterrows():

        comparison_rows.append({
            "Architecture": architecture,
            "Method": row["Method"],
            "Brier_difference": (
                row["Brier_mean"] - baseline["Brier_mean"]
            ),
            "NLL_difference": (
                row["NLL_mean"] - baseline["NLL_mean"]
            ),
            "Adaptive_ECE_difference": (
                row["Adaptive_ECE_mean"]
                - baseline["Adaptive_ECE_mean"]
            )
        })

comparison_df = pd.DataFrame(comparison_rows)

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------

partition_df.to_csv(
    EVAL_DIR / "evaluation_v3_corrected_error_calibration_foldwise.csv",
    index=False
)

bins_df.to_csv(
    EVAL_DIR / "evaluation_v3_corrected_error_calibration_bins.csv",
    index=False
)

summary.to_csv(
    EVAL_DIR / "evaluation_v3_corrected_error_calibration_summary.csv",
    index=False
)

comparison_df.to_csv(
    EVAL_DIR / "evaluation_v3_corrected_error_calibration_comparison.csv",
    index=False
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("\n" + "=" * 90)
print("CORRECTED ERROR-PROBABILITY CALIBRATION")
print("Lower Brier, NLL and ECE are better.")
print("=" * 90)

print(summary.round(6).to_string(index=False))

print("\n" + "=" * 90)
print("DIFFERENCE FROM CONFIDENCE BASELINE")
print("Negative differences favour the reflection method.")
print("=" * 90)

print(comparison_df.round(6).to_string(index=False))

print("\nCALIBRATION ANALYSIS COMPLETE")

## 11. Qualitative Grad-CAM inspection

Twelve MobileNetV3-Small OOF examples are selected across the three folds: high-confidence correct, low-confidence correct, high-confidence incorrect, and low-confidence incorrect. The selected examples come from one held-out split seed to avoid duplicate image instances.

Grad-CAM is used only for qualitative inspection. The visualizations do not establish biological validity, lesion localization accuracy, or causal explanations. Guided Grad-CAM outputs are retained as exploratory visual material and are not used in the numerical reliability score.

In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt

from PIL import Image
from torchvision import datasets, transforms

# ============================================================
# 1. INSTALL AND IMPORT GRAD-CAM
# ============================================================

try:
    from pytorch_grad_cam import GradCAM, GuidedBackpropReLUModel
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image, deprocess_image

    print("Grad-CAM package already available.")

except ImportError:
    print("Installing grad-cam...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "grad-cam"
    ])

    from pytorch_grad_cam import GradCAM, GuidedBackpropReLUModel
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    from pytorch_grad_cam.utils.image import show_cam_on_image, deprocess_image

    print("Grad-CAM installation successful.")

# ============================================================
# 2. PATHS AND REPRODUCIBILITY
# ============================================================

PROJECT_DIR = Path("/kaggle/working/plant_disease_experiments")
RESULTS_DIR = PROJECT_DIR / "results"
MODEL_DIR = PROJECT_DIR / "models"

DATASET_DIR = Path(
    "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage"
)

TEST_FILE = RESULTS_DIR / "evaluation_v3_corrected" / "evaluation_v3_corrected_test_predictions.csv"

OUTPUT_DIR = RESULTS_DIR / "guided_gradcam"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# ============================================================
# 3. LOAD DATASET AND OOF PREDICTIONS
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

dataset = datasets.ImageFolder(
    root=str(DATASET_DIR),
    transform=transform
)

test_df = pd.read_csv(TEST_FILE)

# Restrict Grad-CAM analysis to the MobileNetV3-Small baseline
# Use one of the two held-out split seeds to avoid duplicate image
# instances in the qualitative sample selection.
GRADCAM_SPLIT_SEED = 20261109

test_df = test_df[
    (test_df["Architecture"] == "MobileNetV3-Small")
    & (test_df["Split_seed"] == GRADCAM_SPLIT_SEED)
].copy()

print("Filtered test predictions:", test_df.shape)
print("\nRows per fold and method:")
display(test_df.groupby(["Fold", "Method"]).size().unstack())

assert len(test_df) == 3 * 1720 * 8
assert set(test_df["Fold"].unique()) == {1, 2, 3}
assert test_df["Image_index"].nunique() == 5160

with np.load(
    RESULTS_DIR / "baseline_oof.npz",
    allow_pickle=False
) as z:
    oof_logits = z["oof_logits"].astype(np.float64)
    oof_labels = z["oof_labels"].astype(np.int64)
    oof_predictions = z["oof_predictions"].astype(np.int64)
    oof_folds = z["oof_folds"].astype(np.int64)

assert len(dataset) == 20638
assert len(dataset.classes) == 15
assert len(test_df) == 3 * 1720 * 8

# Softmax confidence from the saved OOF logits.
shifted = oof_logits - oof_logits.max(axis=1, keepdims=True)
exp_logits = np.exp(shifted)
probabilities = exp_logits / exp_logits.sum(axis=1, keepdims=True)

confidence = probabilities.max(axis=1)

# ============================================================
# 4. SELECT FOUR REPRESENTATIVE HELD-OUT SAMPLES PER FOLD
# ============================================================

selected_samples = []

for fold in [1, 2, 3]:

    fold_rows = test_df[
        (test_df["Architecture"] == "MobileNetV3-Small")
        & (test_df["Fold"] == fold)
        & (test_df["Method"] == "Baseline confidence")
    ]

    test_indices = fold_rows["Image_index"].to_numpy(dtype=int)

    assert len(test_indices) == 1720
    assert np.all(oof_folds[test_indices] == fold)

    correct = test_indices[
        oof_predictions[test_indices] == oof_labels[test_indices]
    ]

    incorrect = test_indices[
        oof_predictions[test_indices] != oof_labels[test_indices]
    ]

    assert len(correct) > 0
    assert len(incorrect) > 0

    # Correct prediction with highest confidence.
    high_conf_correct = correct[
        np.argmax(confidence[correct])
    ]

    # Correct prediction with lowest confidence.
    low_conf_correct = correct[
        np.argmin(confidence[correct])
    ]

    # Incorrect prediction with highest confidence.
    high_conf_error = incorrect[
        np.argmax(confidence[incorrect])
    ]

    # Incorrect prediction with lowest confidence.
    low_conf_error = incorrect[
        np.argmin(confidence[incorrect])
    ]

    choices = [
        ("High-confidence correct", high_conf_correct),
        ("Low-confidence correct", low_conf_correct),
        ("High-confidence error", high_conf_error),
        ("Low-confidence error", low_conf_error),
    ]

    for category, index in choices:

        selected_samples.append({
            "Fold": fold,
            "image_index": int(index),
            "Sample_type": category,
            "True_class": dataset.classes[int(oof_labels[index])],
            "OOF_predicted_class": dataset.classes[
                int(oof_predictions[index])
            ],
            "OOF_confidence": float(confidence[index]),
            "OOF_correct": bool(
                oof_predictions[index] == oof_labels[index]
            ),
        })

selection_df = pd.DataFrame(selected_samples)

assert len(selection_df) == 12
assert selection_df["image_index"].nunique() == 12

print("\nSELECTED TEST SAMPLES")
print(selection_df.to_string(index=False))

# ============================================================
# 5. CHECKPOINT LOADER
# ============================================================

def load_checkpoint(fold):

    checkpoint_path = MODEL_DIR / f"mobilenet_fold_{fold}.pth"

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True
    )

    if isinstance(checkpoint, dict):
        for key in ["model_state_dict", "state_dict", "model"]:
            if key in checkpoint and isinstance(checkpoint[key], dict):
                checkpoint = checkpoint[key]
                break

    if all(k.startswith("module.") for k in checkpoint):
        checkpoint = {
            k.replace("module.", "", 1): v
            for k, v in checkpoint.items()
        }

    model = torchvision.models.mobilenet_v3_small(weights=None)
    model.classifier[3] = nn.Linear(
        model.classifier[3].in_features, 15
    )

    model.load_state_dict(checkpoint, strict=True)
    model.to(device)
    model.eval()

    return model


# ============================================================
# 6. GENERATE GRAD-CAM AND GUIDED GRAD-CAM
# ============================================================

manifest_rows = []

for fold in [1, 2, 3]:

    print(f"\nProcessing fold {fold}...")

    model = load_checkpoint(fold)

    # Last convolutional feature block, verified in preflight.
    target_layers = [model.features[-1]]

    cam = GradCAM(
        model=model,
        target_layers=target_layers
    )

    guided_backprop = GuidedBackpropReLUModel(
        model=model,
        device=device
    )

    fold_samples = selection_df[
        selection_df["Fold"] == fold
    ]

    for _, sample in fold_samples.iterrows():

        index = int(sample["image_index"])
        category = sample["Sample_type"]

        image_path = dataset.samples[index][0]
        original_image = Image.open(image_path).convert("RGB")

        # Same resized image used for model preprocessing.
        resized_image = transforms.Resize((224, 224))(original_image)

        rgb_image = (
            np.asarray(resized_image).astype(np.float32) / 255.0
        )

        input_tensor, _ = dataset[index]
        input_tensor = input_tensor.unsqueeze(0).to(device)

        # Verify the checkpoint prediction against saved OOF output.
        with torch.no_grad():
            logits = model(input_tensor)
            probabilities_current = torch.softmax(logits, dim=1)

            predicted_class = int(
                probabilities_current.argmax(dim=1).item()
            )

            model_confidence = float(
                probabilities_current.max().item()
            )

        expected_prediction = int(oof_predictions[index])

        assert predicted_class == expected_prediction, (
            f"Fold {fold}, image {index}: checkpoint prediction "
            "does not match saved OOF prediction."
        )

        # Explain the predicted class, even when it is incorrect.
        targets = [ClassifierOutputTarget(predicted_class)]

        grayscale_cam = cam(
            input_tensor=input_tensor,
            targets=targets
        )[0]

        gradcam_overlay = show_cam_on_image(
            rgb_image,
            grayscale_cam,
            use_rgb=True
        )

        guided_backprop_image = guided_backprop(
            input_tensor,
            target_category=predicted_class
        )

        # Guided Grad-CAM = Grad-CAM mask * Guided Backprop.
        guided_gradcam = deprocess_image(
            grayscale_cam[:, :, None] * guided_backprop_image
        )

        true_name = dataset.classes[int(oof_labels[index])]
        predicted_name = dataset.classes[predicted_class]

        # ----------------------------------------------------
        # Save a three-panel explanation.
        # ----------------------------------------------------

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(rgb_image)
        axes[0].set_title("Original image")

        axes[1].imshow(gradcam_overlay)
        axes[1].set_title("Grad-CAM overlay")

        axes[2].imshow(guided_gradcam)
        axes[2].set_title("Guided Grad-CAM")

        for ax in axes:
            ax.axis("off")

        fig.suptitle(
            f"Fold {fold} | {category}\n"
            f"True: {true_name}\n"
            f"Predicted: {predicted_name} | "
            f"Confidence: {model_confidence:.4f}",
            fontsize=11
        )

        fig.tight_layout()

        safe_category = (
            category.lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        output_path = OUTPUT_DIR / (
            f"fold_{fold}_{safe_category}_image_{index}.png"
        )

        fig.savefig(
            output_path,
            dpi=180,
            bbox_inches="tight"
        )

        plt.close(fig)

        manifest_rows.append({
            **sample.to_dict(),
            "Model_predicted_class": predicted_name,
            "Model_confidence": model_confidence,
            "Visualization_file": str(output_path),
            "Prediction_matches_OOF": True,
        })

        print(
            f"Saved: {output_path.name} | "
            f"{true_name} -> {predicted_name}"
        )

    # Release fold-specific model and hooks before next fold.
    del cam
    del guided_backprop
    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 7. SAVE MANIFEST AND CONTACT SHEET
# ============================================================

manifest = pd.DataFrame(manifest_rows)

manifest_path = OUTPUT_DIR / "guided_gradcam_manifest.csv"
manifest.to_csv(manifest_path, index=False)

# Contact sheet containing all 12 cases.
fig, axes = plt.subplots(3, 4, figsize=(20, 15))

for ax, (_, row) in zip(axes.flat, manifest.iterrows()):

    image = plt.imread(row["Visualization_file"])
    ax.imshow(image)
    ax.set_title(
        f"Fold {row['Fold']} | {row['Sample_type']}\n"
        f"{row['True_class']} -> {row['Model_predicted_class']}",
        fontsize=8
    )
    ax.axis("off")

fig.tight_layout()

contact_path = OUTPUT_DIR / "guided_gradcam_contact_sheet.png"
fig.savefig(contact_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

# ============================================================
# 8. FINAL VERIFICATION
# ============================================================

assert len(manifest) == 12
assert manifest["Prediction_matches_OOF"].all()
assert all(Path(p).exists() for p in manifest["Visualization_file"])

print("\n" + "=" * 75)
print("GUIDED GRAD-CAM GENERATION COMPLETE")
print("=" * 75)

print("Visualizations:", len(manifest))
print("Manifest:", manifest_path)
print("Contact sheet:", contact_path)
print("Output directory:", OUTPUT_DIR)

print("\nFinal manifest:")
print(
    manifest[
        ["Fold", "Sample_type", "True_class",
         "Model_predicted_class", "Model_confidence"]
    ].to_string(index=False)
)

## 12. Interpretation and limitations

The experiments show that EfficientNet-B0 has the highest classification accuracy on this selected PlantVillage subset, while prediction confidence is already a strong error-ranking baseline. The tested combinations of confidence, entropy, and familiarity do not demonstrate a consistent improvement over that baseline.

Important limitations:

- The evaluation uses a selected PlantVillage subset rather than independently collected field images.
- The dataset is class-imbalanced.
- Only the final classifier layer is trained; pretrained parameter tensors are frozen, although BatchNorm running statistics may update in training mode.
- Generalization to field conditions, unseen diseases, and external datasets has not been established.
- Prototype familiarity is not a calibrated probability of correctness.
- Grad-CAM analysis is qualitative and does not use lesion-level annotations.
- The current prototype flags potentially unreliable predictions but does not correct them or learn from user feedback.

Continual learning and verified feedback integration remain future work, not completed PBL-III capabilities.

## Output location

All generated model checkpoints, OOF arrays, evaluation CSVs, manifests, and Grad-CAM images are written under:

```text
/kaggle/working/plant_disease_experiments/
```

The GitHub repository contains only selected research summaries and a Grad-CAM contact sheet to keep the public repository lightweight.